# Local Pipeline 
This pipeline creates and/or updates the tables 'cities', 'populations', and 'airports' in the SQL database 'gans'.  
See 'flights' and 'weather' for containerised, scheduled functions to update flight and weather information.

It is organised in 3 parts:
1. Set up: import functions, set up connection to sql instance, define cities to search
2. Create an empty SQL database to store data (external: using MySQL)
3. Scrape city data (city name, latitude, longitude) from Wikipedia, and airport data () using API; format, and populate SQL tables 'cities' and 'airports' with the resulting dataframes

___
### 1. Set up

In [2]:
#Import functions
from functions import connection_string_cloud, create_gans_tables

In [3]:
#Set up connection to sql instance
connection_string = connection_string_cloud()

In [4]:
#Cities to search
cities = ['Berlin', 'Hamburg', 'Munich']

___
### 2. Create empty SQL database to store data (external: using MySQL)


Run in SQL
```sql
DROP DATABASE IF EXISTS gans;

-- Create the database
CREATE DATABASE gans;

-- Use the database
USE gans;

-- Create the 'countries' table
CREATE TABLE cities (
    city_id INT AUTO_INCREMENT,
    city VARCHAR(255) NOT NULL,
    country VARCHAR(255) NOT NULL,
    lat DECIMAL(8,6) NOT NULL,  -- allow for 2 values before the decimal point (up to 90 degrees latitude) and 6 after the decimal point (precision, i.e. total digits = 8)
    `long` DECIMAL(8,6) NOT NULL,
    PRIMARY KEY (city_id)
);

-- Create the 'populations' table
CREATE TABLE populations (
    city_id INT,
    `population` INT,
    population_date VARCHAR(50),
    retrieved DATETIME,
	FOREIGN KEY (city_id) REFERENCES cities(city_id)
);


-- Create the 'weathers' table
CREATE TABLE weathers (
    city_id INT,
    `date` DATETIME,
    `day` VARCHAR(9),
    `description` VARCHAR(255),
    temp_C DECIMAL(4,2),
    feels_like_C DECIMAL(4,2),
    precipitation_prob DECIMAL(5,3),
    humidity_pct INT,
    cloudiness_pct INT,
    wind_speed_meters_per_s DECIMAL (5,2),
    wind_direction VARCHAR(2),
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);

-- Create the 'airports' table
CREATE TABLE airports (
    city_id INT,
    icao VARCHAR(4),
    name VARCHAR(255),
    PRIMARY KEY (icao),
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);

-- Create the 'flights' table
CREATE TABLE flights (
    flight_id INT AUTO_INCREMENT,
    icao VARCHAR(4),
    `status` VARCHAR(255),
    scheduled_time DATETIME,
    revised_time DATETIME,
    origin_icao VARCHAR(4),
    origin_name VARCHAR(255),
    origin_tz VARCHAR(255),
    PRIMARY KEY (flight_id),
    FOREIGN KEY (icao) REFERENCES airports(icao)
);
```

___
### 3. Scrape city, populations and airport data (slowly changing dimensions) and send to sql

In [5]:
create_gans_tables(cities, connection_string)